# Query Generation

### Setup & imports

In [2]:
# Setup & imports
from pathlib import Path
import ntpath
from datetime import datetime
import sys
import pandas as pd
import re
import json

### Setup & imports

In [ ]:
# Setup & imports

import os
import base64
from openai import AzureOpenAI

endpoint = os.getenv("ENDPOINT_URL", "")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", "")  

# Initialize Azure OpenAI client with key-based authentication",
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2025-01-01-preview",
)


In [ ]:

#prompt builder

def build_prompt(passages, system_prompt):

    chat_prompt = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_prompt
                }
            ]
        },
        {
            "role": "user",
            "content": passages
        }
    ]
    return chat_prompt


In [13]:
# Modeling & evaluation
def openai_api(lines, system_prompt):
    # Include speech result if speech is enabled
    messages = build_prompt(lines, system_prompt)
    
    # Generate the completion
    completion = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=2000,
        temperature=0.7,
        top_p=0.95,
        frequency_penalty=0,
        presence_penalty=0,
        stop=None,
        stream=False
    )
    
    return completion.choices[0].message.content

In [5]:
# Computation
system_prompt = '''Generate a relevant question-answer pair based on a conversation referencing an image, ensuring the question explicitly ties to a specific date, individual, or notable event discussed.

- Avoid directly mentioning "image" in the question. Focus on natural phrasing like "What did the living room look like in September?" instead of "What was visible in the image of the living room from September?"
- The answer should be a single, concise sentence articulating the description or detail clearly.

# Steps:
1. Identify a notable date, named individual, or significant event from the conversation to anchor the question.
2. Frame the question based on the referenced topic while avoiding direct mention of the word "image."
3. Write a short, concise, and factual answer derived from visual or contextual clues inferred from the conversation. Be direct and clear.

# Output Format:
JSON:
{
  "question": "[Generated question based on the conversation]",
  "answer": "[Concise answer based on referenced visual details]"
}

# Examples:
### Example 1:
**Conversation snippet**: "Emily mentioned that the living room was repainted in September. The photo we received later showed it fully redecorated with green accents."
**Output**:
```json
{
  "question": "What did the living room look like in September?",
  "answer": "The living room had been repainted with green accents."
}
```

### Example 2:
**Conversation snippet**: "During the charity event in June, Mark stood by the large banner displaying the slogan for the fundraiser."
**Output**:
```json
{
  "question": "Where was Mark during the charity event in June?",
  "answer": "Mark stood by the large banner with the fundraiser slogan."
}
```'''

In [30]:
# Data loading
from pathlib import Path  
  
# Specify the directory you want to start from  
start_directory = Path('/home/ec2-user/repos/msc/conversations')  

qa_pairs = []
  
# Iterate through all .txt files in the directory and subdirectories  
for txt_file in start_directory.rglob("*.txt"):  # rglob searches recursively 
    with open(txt_file, 'r') as file:  
        # Read the contents of the file into a string  
        file_contents = file.read()   

        response = openai_api(file_contents, system_prompt)
        data = json.loads(response[8:-4])
        result = list(data.values()) 
        qa_pairs.append(result)

In [31]:
# Computation
qa_pairs_df = pd.DataFrame(qa_pairs, columns=["question", "answer"])

In [33]:
# Computation
qa_pairs_df.to_json("questionanswers.json", orient="records", indent=4) 